# FoodLensVN — Gradio Demo (A1 / A2 / B1 / B2)

Launches `app/demo.py` on Kaggle and exposes it via a free Cloudflare tunnel — no ngrok, no auth token. Public URL appears **after** every model is loaded into the subprocess, so the first viewer click doesn't stall on a cold load.

Why subprocess + cloudflared (not `gradio --share`):
- The notebook process never imports `torch`. CUDA contexts are sticky — once the notebook touches a GPU, that memory is reserved for the kernel's lifetime, and the demo subprocess fights it for VRAM (T4 OOMs).
- `gradio --share` is flaky on Kaggle; cloudflared is rock-solid.
- `subprocess.Popen` with stdout redirected to a log file avoids the 64 KB pipe-buffer deadlock that hangs `!uv run python app/demo.py` mid-load.

**Setup before running:**
1. **Settings → Accelerator → GPU** (P100 or T4 ×2).
2. **Settings → Internet → On**.
3. **Add-ons → Secrets → `HF_TOKEN`** (read access).

In [ ]:
# Cell 1: Clone develop or pull latest. Pure git, no torch.
import os, subprocess

REPO_URL = 'https://github.com/tamir39/vqa-viet-project.git'
REPO_DIR = '/kaggle/working/vqa-viet-project'
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'develop', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
os.chdir(REPO_DIR)
print(subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# Cell 2: Install deps via uv (project) + cloudflared client + hf_transfer.
!pip install -q uv hf_transfer
!uv sync --frozen 2>&1 | tail -10

In [ ]:
# Cell 3: HF login via Kaggle secret. No torch import.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print('HF login OK')

In [ ]:
# Cell 4: Build the dataset splits (eval/build artifacts only — no torch, no GPU).
os.environ['FOODLENS_DATA_DIR'] = f'{REPO_DIR}/data/foodlensvn'
os.environ['MPLBACKEND'] = 'Agg'
!uv run python scripts/fetch_dataset.py --dest $FOODLENS_DATA_DIR
!uv run python scripts/build_dataset.py --data-dir $FOODLENS_DATA_DIR --output-dir data/processed --image-variant squared

In [ ]:
# Cell 5: Pull A1/A2 checkpoints + B2 adapter. snapshot_download is pure file I/O —
# no torch import, no CUDA context allocated in this notebook process.
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
for name in ['A1', 'A2', 'B2']:
    snapshot_download(
        repo_id=f'Tamir39/foodlensvn-{name}',
        repo_type='model',
        local_dir=f'reports/{name}',
        token=os.environ['HF_TOKEN'],
        max_workers=4,
        etag_timeout=30,
    )
    print(f'{name}: pulled to reports/{name}/')

# Pre-cache the Qwen2-VL base too — the demo subprocess will use the local cache
# instead of downloading 4 GB while the tunnel is open (which times out).
snapshot_download('Qwen/Qwen2-VL-2B-Instruct',
                  allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model'])
print('Qwen2-VL-2B-Instruct: cached')

In [ ]:
# Cell 6: Download the cloudflared binary (one-time, ~30 MB).
import os, stat, subprocess

BIN = '/kaggle/working/cloudflared'
if not os.path.isfile(BIN):
    subprocess.check_call([
        'wget', '-q', '-O', BIN,
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    ])
    os.chmod(BIN, os.stat(BIN).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
print(subprocess.check_output([BIN, '--version']).decode().strip())

In [ ]:
# Cell 7: Launch app/demo.py as a subprocess with --eager so every track is
# loaded BEFORE gradio binds. stdout/stderr are redirected to a log file —
# Linux pipes are 64 KB, and once full every print() inside the demo blocks
# (model load wedges with no output and no progress).
#
# PYTHONUNBUFFERED + python -u: when stdout is redirected to a file, Python
# block-buffers it (4-8 KB). Gradio's 'Running on local URL' line then sits
# in memory long enough that our readline loop times out. -u forces line
# buffering so every print flushes immediately.
#
# Tail the log live in a separate cell with:  !tail -f /kaggle/working/demo.log
import os, re, subprocess, time

DEMO_LOG = '/kaggle/working/demo.log'
CF_LOG = '/kaggle/working/cloudflared.log'

demo_env = os.environ.copy()
demo_env['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
demo_env['PYTHONUNBUFFERED'] = '1'

demo = subprocess.Popen(
    ['uv', 'run', 'python', '-u', 'app/demo.py', '--eager', '--server-port', '7860'],
    stdout=open(DEMO_LOG, 'w'), stderr=subprocess.STDOUT,
    env=demo_env,
    cwd=REPO_DIR,
)
print(f'[demo] launching (logs: {DEMO_LOG})...')

# Wait until gradio prints its 'Running on ...' line. Eager loading takes a
# few minutes (A1, A2, then NF4 Qwen + LoRA), so allow a generous timeout.
started = False
deadline = time.time() + 600  # 10 min
with open(DEMO_LOG) as f:
    while time.time() < deadline:
        line = f.readline()
        if not line:
            if demo.poll() is not None:
                raise RuntimeError(f'demo subprocess exited early — see {DEMO_LOG}')
            time.sleep(0.5)
            continue
        print(line.rstrip())
        if 'Running on local URL' in line or 'Running on http' in line:
            started = True
            break
if not started:
    raise RuntimeError(f'demo never came up — see {DEMO_LOG}')

# Open the cloudflared tunnel.
print(f'\n[cloudflared] opening tunnel (logs: {CF_LOG})...')
tunnel = subprocess.Popen(
    [BIN, 'tunnel', '--no-autoupdate', '--url', 'http://localhost:7860'],
    stdout=open(CF_LOG, 'w'), stderr=subprocess.STDOUT,
)

public_url = None
deadline = time.time() + 60
with open(CF_LOG) as f:
    while time.time() < deadline:
        line = f.readline()
        if not line:
            time.sleep(0.25)
            continue
        print(line.rstrip())
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            break

if not public_url:
    raise RuntimeError(f'could not parse cloudflared URL — see {CF_LOG}')

print('\n' + '=' * 72)
print(f'  DEMO URL:  {public_url}')
print('=' * 72)
print('\nFollow logs in another cell:')
print(f'  !tail -f {DEMO_LOG}    # demo subprocess (model load, requests)')
print(f'  !tail -f {CF_LOG}      # cloudflared (tunnel events)')
print('\nStop the demo by interrupting this cell or stopping the kernel.')

## Stopping the demo

- **Interrupt the cell above** (◼ button) — this kills both the demo and the tunnel.
- Or **Stop Kernel** from the right sidebar to shut down the whole session.
- Closing the browser tab does **not** stop the session — the tunnel keeps burning your GPU quota until the kernel idles out (~20 min).

## Troubleshooting

- **Tunnel URL prints but page shows error** — wait ~10 s and reload; cloudflared takes a moment to register.
- **OOM during eager load** — switch accelerator to T4 ×2 (30 GB combined). With one T4 (15 GB), B2 (LoRA on Qwen NF4) + A1 + A2 is tight but should fit.
- **Demo subprocess exits early** — `cat /kaggle/working/demo.log` for the actual error.